In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
# We use the raw subnet directory to get all columns (not just n_bytes)
DATA_DIR = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"

# Output for the plot
SAVE_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Feature_Analysis")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. LOAD ONE RAW FILE ---
csv_files = list(SUBNET_DIR.glob("*.csv"))

if not csv_files:
    print("No CSV files found! Check your path.")
else:
    target_file = next((f for f in csv_files if "9.csv" in f.name), csv_files[0])
    print(f"Analyzing features for: {target_file.name}")

    df_raw = pd.read_csv(target_file)

    # Check what columns we actually have
    print(f"Available Columns: {df_raw.columns.tolist()}")

    # Select the traffic metrics
    target_cols = ['n_bytes', 'n_packets', 'n_flows']

    # Filter to only existing columns
    cols_to_plot = [c for c in target_cols if c in df_raw.columns]

    if len(cols_to_plot) > 1:
        # --- 2. COMPUTE CORRELATION ---
        corr_matrix = df_raw[cols_to_plot].corr()
        print("\n=== CORRELATION MATRIX ===")
        print(corr_matrix)

        # --- 3. PLOT HEATMAP ---
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=0, vmax=1, fmt=".4f")
        plt.title(f"Feature Correlation (Subnet {target_file.stem})")
        plt.tight_layout()
        plt.savefig(SAVE_DIR / "feature_correlation_heatmap.png")
        plt.show()

        # --- 4. PLOT SCATTER (BYTES vs PACKETS) ---
        # This visualizes the linearity
        if 'n_bytes' in cols_to_plot and 'n_packets' in cols_to_plot:
            plt.figure(figsize=(8, 6))
            sns.scatterplot(data=df_raw, x='n_bytes', y='n_packets', alpha=0.5, edgecolor=None)
            plt.title("Linearity Check: Bytes vs. Packets")
            plt.xscale('log')
            plt.yscale('log')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(SAVE_DIR / "bytes_vs_packets_scatter.png")
            plt.show()

        print("Feature analysis plots saved.")
    else:
        print("Could not find multiple traffic columns to correlate.")

In [ ]:
#import pandas as pd
#import numpy as np
#from pathlib import Path
#
## --- CONFIG ---
#DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
#SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"
#
#candidates = []
#
#print("Scanning subnets for Attack Signatures...")
#
#for csv_path in SUBNET_DIR.glob("*.csv"):
#    sid = csv_path.stem
#    df = pd.read_csv(csv_path)
#
#    # Filter for active traffic to avoid division by zero
#    df = df[df['n_packets'] > 10].copy()
#    if len(df) < 50: continue
#
#    # --- METRIC 1: THE FLOOD SCORE (DDoS) ---
#    # High Packets + Low Duration + Low TCP Ratio
#    # We look for the Max value of this combination
#    # (Using +1 to avoid div by zero)
#    flood_signal = (df['n_packets'] / (df['avg_duration'] + 0.1)) * (1 - df['tcp_udp_ratio_packets'])
#    max_flood = flood_signal.max()
#
#    # --- METRIC 2: THE SCAN SCORE (Port Scan) ---
#    # High Flows + High Destination Diversity (IPs or Ports)
#    # Scanners create many flows to many destinations
#    if 'sum_n_dest_ip' in df.columns:
#        scan_signal = df['n_flows'] * df['sum_n_dest_ip']
#    else:
#        scan_signal = df['n_flows'] # Fallback if col missing
#    max_scan = scan_signal.max()
#
#    candidates.append({
#        'sid': sid,
#        'max_flood_score': max_flood,
#        'max_scan_score': max_scan
#    })
#
## --- RESULTS ---
#results = pd.DataFrame(candidates)
#
## 1. Top Potential DDoS Candidates
#print("\nTOP CANDIDATES: POTENTIAL DDoS (UDP Flood)")
#ddos_cand = results.sort_values(by='max_flood_score', ascending=False).head(3)
#print(ddos_cand[['sid', 'max_flood_score']])
#
## 2. Top Potential Scanning Candidates
#print("\nTOP CANDIDATES: POTENTIAL SCANS (High Flows/Destinations)")
#scan_cand = results.sort_values(by='max_scan_score', ascending=False).head(3)
#print(scan_cand[['sid', 'max_scan_score']])
#
#print("\nRecommendation: Pick the #1 from either list and run the Forensic Plot code on it.")

In [ ]:
#import matplotlib.pyplot as plt
#import pandas as pd
#from pathlib import Path
#
## --- CONFIG ---
#DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
#SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"
#TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"
#TARGET_SID = '171'  # The "Attack" Candidate
#
## 1. Load Time Mapping
#print("Loading time mapping...")
#times_df = pd.read_csv(TIMES_PATH)
#times_df["time"] = pd.to_datetime(times_df["time"], errors="coerce")
#if times_df["time"].dt.tz is not None:
#    times_df["time"] = times_df["time"].dt.tz_localize(None)
#times_df["id_time"] = times_df["id_time"].astype(int)
#
## 2. Load Subnet 171
#print(f"Loading raw data for Subnet {TARGET_SID}...")
#file_path = list(SUBNET_DIR.glob(f"*{TARGET_SID}.csv"))[0]
#df = pd.read_csv(file_path)
#
## 3. Merge Dates
#df["id_time"] = df["id_time"].astype(int)
#df = df.merge(times_df, on="id_time", how="left")
#df = df.sort_values("time").set_index("time")
#
## 4. Find the Attack Window
## We look for the moment where the "Flood Score" was highest to zoom in
## Score = Packets / Duration (Roughly)
#df['attack_score'] = df['n_packets'] / (df['avg_duration'] + 0.1)
#peak_time = df['attack_score'].idxmax()
#
## Zoom in: 1 week before and 1 week after the peak
#start_zoom = peak_time - pd.Timedelta(days=7)
#end_zoom   = peak_time + pd.Timedelta(days=7)
#subset = df.loc[start_zoom:end_zoom].copy()
#
## --- PLOT ---
#print("Generating Attack Forensic Plot...")
#fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
#
## Panel 1: INTENSITY (Packets)
## Attacks are often better visible in Packets than Bytes
#axes[0].plot(subset.index, subset['n_packets'], color='#c0392b', linewidth=1.5)
#axes[0].set_title(f"1. Attack Intensity: Packet Volume (Subnet {TARGET_SID})", fontsize=12, fontweight='bold', loc='left')
#axes[0].set_ylabel("Packets")
#axes[0].grid(True, alpha=0.3)
#
## Panel 2: SPREAD (Destinations)
## Did they target one IP or thousands?
#axes[1].plot(subset.index, subset['sum_n_dest_ip'], color='#8e44ad', linewidth=1.5)
#axes[1].set_title("2. Attack Spread: Distinct Destination IPs", fontsize=12, fontweight='bold', loc='left')
#axes[1].set_ylabel("Dest IPs")
#axes[1].grid(True, alpha=0.3)
#
## Panel 3: BEHAVIOR (Duration)
## DDoS packets are usually instant (Duration ~ 0)
#axes[2].plot(subset.index, subset['avg_duration'], color='#2c3e50', linewidth=1.5)
#axes[2].set_title("3. Traffic Behavior: Average Duration", fontsize=12, fontweight='bold', loc='left')
#axes[2].set_ylabel("Duration (ms)")
#axes[2].grid(True, alpha=0.3)
#
## Panel 4: TYPE (Protocol)
## Did it switch to pure UDP?
#subset['tcp_udp_ratio_packets'] = subset['tcp_udp_ratio_packets'].fillna(0)
#axes[3].plot(subset.index, subset['tcp_udp_ratio_packets'], color='#27ae60', linewidth=1.5)
#axes[3].set_title("4. Protocol Type: TCP/UDP Ratio", fontsize=12, fontweight='bold', loc='left')
#axes[3].set_ylabel("Ratio")
#axes[3].set_ylim(0, 5) # Focus on the low range
#axes[3].grid(True, alpha=0.3)
#
#plt.xlabel("Date")
#plt.tight_layout()
#plt.show()